# Cross validation

Key points:
- Corss-validation estimates how well a model generalizes by repeatedly splitting data into train/test folds
- For classification, preserving class balance often matters
- Different splitters answer different questions:
 - plain K-fold
 - stratified K-fold
 - leave-one-out
 - shuffle-split
 - grouped cross validation

## 1. The core idea

A single train/test split can be noisy because the estimate depends on one accidental split.

Cross-validation improves stability by:
- splitting the data multiple times
- fitting the model repeatedly
- summarizing performance across folds

Typically report:
- mean score
- standard deviation across folds
- possibly the full fold-wise scores

In [52]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_iris, load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (
    KFold, StratifiedKFold, LeaveOneOut, ShuffleSplit, StratifiedShuffleSplit,
    GroupKFold, LeaveOneGroupOut, StratifiedGroupKFold, 
    cross_val_score, cross_validate, GridSearchCV, train_test_split
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.feature_selection import SelectKBest, f_classif

iris = load_iris()
X, y = iris.data, iris.target
clf = LogisticRegression(max_iter=2000)

## 2. The default baseline: K-fold vs stratified K-fold

### K-fold

 - splits the data into k folds
 - each fold serves once as test set
 - good default for regression

### Stratified K-fold
 - keeps class proportions roughly similar across folds
 - usually the better default for classification

Stratification is mainly an engineering fix for unstable class distributions. It helps the estimate behave more sensibly, especially on small or imbalanced datasets.

In [53]:
kfold = KFold(n_splits=5, shuffle=True, random_state=0)
scores_kfold = cross_val_score(clf, X, y, cv=kfold)

skfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
scores_skfold = cross_val_score(clf, X, y, cv=skfold)

print("KFold scores:          ", np.round(scores_kfold, 3))
print("KFold mean ± std:      ", round(scores_kfold.mean(), 3), "±", round(scores_kfold.std(), 3))
print()
print("StratifiedKFold scores:", np.round(scores_skfold, 3))
print("Stratified mean ± std: ", round(scores_skfold.mean(), 3), "±", round(scores_skfold.std(), 3))


KFold scores:           [1.    0.833 1.    1.    0.933]
KFold mean ± std:       0.953 ± 0.065

StratifiedKFold scores: [0.967 0.967 0.967 0.967 0.933]
Stratified mean ± std:  0.96 ± 0.013


On balanced data like iris, both may work fine if you shuffle first.

But if the data are ordered by class, plain `KFold` can be misleading. 

In [54]:
ordered_idx = np.argsort(y)
X_ordered = X[ordered_idx]
y_ordered = y[ordered_idx]

bad_kfold = KFold(n_splits=3, shuffle=False)
bad_scores = cross_val_score(clf, X_ordered, y_ordered, cv=bad_kfold)

good_kfold = StratifiedKFold(n_splits=3, shuffle=False)
good_scores = cross_val_score(clf, X_ordered, y_ordered, cv=good_kfold)

print("Non-shuffled KFold on ordered labels:     ", np.round(bad_scores, 3))
print("StratifiedKFold on ordered labels:        ", np.round(good_scores, 3))

Non-shuffled KFold on ordered labels:      [0. 0. 0.]
StratifiedKFold on ordered labels:         [0.98 0.98 0.96]


## 3. `cross_val_score` vs `cross_validate`

Use:

- `cross_val_score` for one quick metric
- `cross_validate` for richer output

`cross_validate` is often better for series work because you can:

- request several metrics
- inspect fit time/score time
- optionally return fitted estimators or train scores

In [55]:
scoring = {
    "accuracy": "accuracy",
    "f1_macro": "f1_macro"
}
cv_results = cross_validate(
    clf, X, y,
    cv=skfold,
    scoring=scoring,
    return_train_score=True
)
pd.DataFrame(cv_results)

,fit_time,score_time,test_accuracy,train_accuracy,test_f1_macro,train_f1_macro
0,0.005584,0.001036,0.966667,0.966667,0.966583,0.966646
1,0.004330,0.001130,0.966667,0.975000,0.966583,0.974996
2,0.003771,0.000853,0.966667,0.983333,0.966583,0.983323
3,0.004157,0.000959,0.966667,0.975000,0.966583,0.974996
4,0.004664,0.000746,0.933333,0.991667,0.933333,0.991665


## 4. Leave-one-out(LOO)

Each split uses:
- 1 observation as test set
- all remainig observations as training set

When it can make sense
- very small datasets
- when you want to use nearly all observations for training in each iteration

Downsides:
- computationally expensive
- high variance estimate
- often not better in practice than a repeated or stratified K-fold estimate


In [56]:
loo = LeaveOneOut()
scores_loo = cross_val_score(clf, X, y, cv=loo)

print("Number of LOO iterations:", len(scores_loo))
print("Mean LOO accuracy:", round(scores_loo.mean(), 4))
print("Std of LOO scores:", round(scores_loo.std(), 4))


Number of LOO iterations: 150
Mean LOO accuracy: 0.9667
Std of LOO scores: 0.1795


LOO is more of a special-case tool than a default choice. For many practical datasets, 5-fold or 10-fold CV is more efficient and easier to interpret. 

## 5. Shuffle-Split

### Idea
Repeatedly draw random train/test splits

Why it is useful

- control train and test sizes directly
- control how many random splits to run
- useful when you do not need each sample to appear exactly once in a test fold
- handy for large datasets because you can subsample


In [57]:
shuffle_split = ShuffleSplit(
    n_splits=10,
    train_size=0.5,
    test_size=0.5,
    random_state=0
)

scores_ss = cross_val_score(clf, X, y, cv=shuffle_split)

print("ShuffleSplit scores:", np.round(scores_ss, 3))
print("Mean ± std:", round(scores_ss.mean(), 3), "±", round(scores_ss.std(), 3))



ShuffleSplit scores: [0.933 0.933 0.987 0.947 0.96  0.947 0.973 1.    0.96  0.987]
Mean ± std: 0.963 ± 0.022


For classification, often prefer `StratifiedShuffleSplit`. This combines:
- random repeated splitting
- approximate preservation of class proportions

In [58]:
sss = StratifiedShuffleSplit(
    n_splits=10,
    train_size=0.5,
    test_size=0.5,
    random_state=0
)

scores_sss = cross_val_score(clf, X, y, cv=sss)

print("StratifiedShuffleSplit scores:", np.round(scores_sss, 3))
print("Mean ± std:", round(scores_sss.mean(), 3), "±", round(scores_sss.std(), 3))


StratifiedShuffleSplit scores: [0.96  0.96  0.973 0.96  0.96  0.947 0.947 0.96  0.987 0.947]
Mean ± std: 0.96 ± 0.012


## 6. Grouped cross-validation: the antileakage splitter

If multiple rows come from the same unit, ordinary CV can leak information.

Examples:
- several images from the same person
- repeated visits from the same patient
- multiple utterances from the same speaker
- several transactions from the same customer
- many students from the same classroom
- multiple time windows from the same device

If the same group appears in both training and test folds, performance can look unrealistically good.

### `GroupKFold`

Keeps groups non-overlapping across train and test folds

Use it when:
- groups matter
- and your main goal is avoiding leakage across related samples

In [59]:
X_group = np.arange(24).reshape(12, 2)
y_group = np.array([0,0,0,1,1,1,1,0,0,1,1,1])
groups = np.array([1,1,1,2,2,2,2,3,3,4,4,4])

gkf = GroupKFold(n_splits=4)

for fold, (train_idx, test_idx) in enumerate(gkf.split(X_group, y_group, groups=groups), start=1):
    print(f"Fold {fold}")
    print("  train groups:", np.unique(groups[train_idx]))
    print("  test groups: ", np.unique(groups[test_idx]))



Fold 1
  train groups: [1 3 4]
  test groups:  [2]
Fold 2
  train groups: [1 2 3]
  test groups:  [4]
Fold 3
  train groups: [2 3 4]
  test groups:  [1]
Fold 4
  train groups: [1 2 4]
  test groups:  [3]


In [60]:
from sklearn.datasets import make_blobs

X, y = make_blobs(n_samples=12, random_state=0)
labels = [0,0,0,1,1,1,1,2,2,3,3,3]

logreg = LogisticRegression()
gkf = GroupKFold(n_splits=4)

cross_val_score(logreg, X, y, groups=labels, cv=gkf)

array([0.75      , 0.66666667, 0.66666667, 1.        ])

### `LeaveOneGroupOut`

A stricter grouped strategy:
- each split leaves one entire group out for testing

Useful when:
- each group is a meaningful uit
- want a direct answer to: how well do we generalize to a completely new group?

In [61]:
logo = LeaveOneGroupOut()

for fold, (train_idx, test_idx) in enumerate(logo.split(X_group, y_group, groups=groups), start=1):
    print(f"Fold {fold}: test group = {np.unique(groups[test_idx])}")


Fold 1: test group = [1]
Fold 2: test group = [2]
Fold 3: test group = [3]
Fold 4: test group = [4]


In [62]:

cross_val_score(logreg, X, y, groups=labels, cv=logo)

array([0.66666667, 0.75      , 1.        , 0.66666667])

### `StratifiedGroupKFold`

It aims to satisfy two objectives simultaneously:
- preserving class balance across folds
- ensuring non-overlapping groups

>This method can be seen as balancing two competing structures in the data:
group-level dependency and class distribution. When these conflict, preserving
group structure takes priority, making stratification approximate rather than exact.

Use it when:
- the task is classification
- the data contains grouped or dependent observations
- maintaining class distribution is important (e.g., class imbalance)

In [63]:
sgkf = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=0)

for fold, (train_idx, test_idx) in enumerate(sgkf.split(X_group, y_group, groups=groups), start=1):
    print(f"Fold {fold}")
    print("  train groups:", np.unique(groups[train_idx]))
    print("  test groups: ", np.unique(groups[test_idx]))
    print("  test class counts:", np.bincount(y_group[test_idx]))


Fold 1
  train groups: [1 3 4]
  test groups:  [2]
  test class counts: [0 4]
Fold 2
  train groups: [1 2]
  test groups:  [3 4]
  test class counts: [2 3]
Fold 3
  train groups: [2 3 4]
  test groups:  [1]
  test class counts: [3]


## 7. The biggest practical mistake: preprocessing outside CV

A splitter is only part of the story.
If you scale, impute, select features, or do PCA before cross-validation, information from the test folds leaks into training.

### Correct rule

Everything that learns from the data must happen inside the pipeline.

In [64]:
cancer = load_breast_cancer()
Xc, yc = cancer.data, cancer.target

#wrong
selector = SelectKBest(score_func=f_classif, k=10)
Xc_selected_wrong = selector.fit_transform(Xc, yc)
wrong_scores = cross_val_score(LogisticRegression(max_iter=5000), Xc_selected_wrong, yc)

#correct
pipeline = make_pipeline(
    StandardScaler(),
    SelectKBest(score_func=f_classif, k=10),
    LogisticRegression(max_iter=5000)
)
right_scores = cross_val_score(pipeline, Xc, yc, cv=5)

print("Potentially optimistic (wrong workflow):", np.round(wrong_scores, 3), "mean =", round(wrong_scores.mean(), 3))
print("Leakage-safe pipeline workflow:         ", np.round(right_scores, 3), "mean =", round(right_scores.mean(), 3))


Potentially optimistic (wrong workflow): [0.921 0.947 0.965 0.939 0.947] mean = 0.944
Leakage-safe pipeline workflow:          [0.904 0.947 0.982 0.965 0.956] mean = 0.951


In cross-validation with a pipeline, the data is split first. Then each preprocessing step is fitted only on the training fold. The learned preprocessing parameters, such as selected features or scaling statistics, are subsequently applied to the test fold via transform, without refitting on the test data.

## 8. Choosing the splitter: a practical decision table

### Use `KFold`
- regression
- no strong grouping issue
- roughly IID data
- often with `shuffle=True, random_state=...`

### Use `StratifiedKFold`
- classification
- class proportions matter
- no grouping issue

### Use `LeaveOneOut`
- very small dataset
- special-case analysis
- accept high computational cost

### Use `ShuffleSplit`/`StratifiedShuffleSplit`
- prepeated random resampling
- flexible train/test sizes
- large datasets or quick experiments

### Use `GroupKFold`/`LeaveOneGroupOut`
- repeated measurements within groups
- patient/speaker/person/customer leakage risk

### Use `StratifiedGroupKFold`
- classification + groups + class balance concerns

## 9. Cross-validation is for estimation;grid search is for tuning

- CV estimates generalization performance
- grid search chooses hyperparameters
- grid search itself should usually use CV internally
- if you want an unbiased evaluation of the whole tuning process, use nested CV

In [65]:
X, y = iris.data, iris.target

pipe_svc = make_pipeline(StandardScaler(), SVC())

param_grid = {
    "svc__C": [0.1, 1, 10],
    "svc__gamma": ["scale", 0.01, 0.1, 1]
}

grid = GridSearchCV(
    pipe_svc,
    param_grid=param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=0),
    scoring="accuracy",
    n_jobs=1
)

grid.fit(X, y)

print("Best params:", grid.best_params_)
print("Best mean CV score:", round(grid.best_score_, 3))


Best params: {'svc__C': 10, 'svc__gamma': 0.01}
Best mean CV score: 0.96


## 10. Short comparison table

| Splitter | Best for | Main strength | Main caution |
|---|---|---|---|
| KFold | regression, IID data | simple baseline | poor for ordered/imbalanced classification |
| StratifiedKFold | classification | preserves class proportions | not a substitute for good sampling design |
| LeaveOneOut | very small data | maximal training data each split | slow, high variance |
| ShuffleSplit | flexible repeated splitting | choose train/test sizes freely | overlapping test sets possible |
| StratifiedShuffleSplit | classification with repeated random splitting | class balance + flexibility | overlapping test sets possible |
| GroupKFold | grouped data | avoids leakage across groups | may not preserve class balance |
| LeaveOneGroupOut | grouped data, stronger generalization question | tests on entirely new groups | can be pessimistic if groups differ a lot |
| StratifiedGroupKFold | grouped classification | balances classes while respecting groups | only approximate balance may be possible |



## 11. Key takeways

### Practical defaults
- Classification: `StratifiedKFold`
- Regression: `KFold(shuffle=True, random_state=...)`
- Grouped data: `GroupKFold` or `StratifiedGroupKFold`
- Tiny dataset: maybe `LeaveOneOut`, but only with a good reason
- Random repeated experiments: `ShuffleSplit` / `StratifiedShuffleSplit`
